# 1. Data Preprocessing, Normalizing: Tiền xử lí dữ liệu, EDA, Chuẩn hóa dữ liệu.

In [ ]:
import json
import torch
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer
from torch.utils.data import Dataset

# 1. Đọc file Dataset MỚI
with open('Gen dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

texts = []
labels = []
for intent in dataset['intents']:
    for pattern in intent['patterns']:
        texts.append(pattern)
        labels.append(intent['tag'])

# 2. Mã hóa Nhãn
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)

# 3. Tokenize bằng BERT
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)

# 4. Đóng gói cho PyTorch
class MentalHealthDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MentalHealthDataset(encodings, encoded_labels)

print(f"✅ Hoàn tất! Số class: {num_classes}")
print(f"📚 Dữ liệu học: {len(texts)} câu (Sử dụng 100% Data)")

# 2. Model Architecture: Xây dựng mô hình.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=num_classes
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"✅ Đã tải thành công lên: {device.type.upper()}")

In [ ]:
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Free: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB reserved")

In [ ]:
free, total = torch.cuda.mem_get_info()
print(f"GPU total: {total / 1e9:.2f} GB")
print(f"GPU free: {free / 1e9:.2f} GB")
print(f"PyTorch allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"PyTorch reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# 3. Model Training: Train Mô hình Chatbot.

In [ ]:
import torch, gc, pickle
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

torch.cuda.empty_cache()
gc.collect()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free / 1e9:.2f} GB / {total / 1e9:.2f} GB")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_classes
)

model.gradient_checkpointing_enable()
model.to(device)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    fp16=True,
    optim="adafactor",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    disable_tqdm=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

model.save_pretrained("./bert_chatbot_model")
tokenizer.save_pretrained("./bert_chatbot_model")

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Saved.")

# 4. Infering & Chatting: Tạo luồng - Chạy thử bot.

In [ ]:
import torch
import random
import json
import pickle
import requests # Dùng để gọi Ollama
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 1. TẢI "CÔ LỄ TÂN" BERT
model = AutoModelForSequenceClassification.from_pretrained("./bert_chatbot_model")
tokenizer = AutoTokenizer.from_pretrained("./bert_chatbot_model")
with open("label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
with open('Gen dataset.json', 'r', encoding='utf-8') as f:
    intents_data = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

bot_name = "Meiritsuu"
print(f"🤖 {bot_name} (BERT + LLM) is ready!~ Press 'quit' to exit~~~.")
print("-" * 50)

# Hàm gửi lệnh cho "Bác sĩ" Llama-3 qua API của Ollama
def ask_llm(user_input, medical_context):
    # Prompt cho mô hình 3B cần ngắn gọn, đi thẳng vào vấn đề
    prompt = f"""You are Meiritsuu, an empathetic mental health AI assistant. 
User says: "{user_input}"
Medical Knowledge to base your answer on: "{medical_context}"

Task: Write a short, caring, and conversational response to the user. 
Rule 1: Use ONLY the provided Medical Knowledge. Do NOT make up facts.
Rule 2: Keep it concise (under 3 sentences)."""

    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "llama3.2",   # <--- CHỈ CẦN THAY ĐỔI TÊN MÔ HÌNH Ở DÒNG NÀY
        "prompt": prompt,
        "stream": False
    }
    response = requests.post(url, json=payload)
    return response.json()['response']

# 2. VÒNG LẶP CHAT
while True:
    sentence = input("Huno: ")
    if sentence.lower() == "quit":
        break

    # --- BƯỚC 1: BERT PHÂN LUỒNG ---
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()
    probs = torch.softmax(logits, dim=1)
    prob = probs[0][predicted_class_id].item()

    if prob > 0.65:
        tag_name = label_encoder.inverse_transform([predicted_class_id])[0]
        
        # Bắt Hard Negatives (Ngoài phạm vi)
        if tag_name == "out_of_scope":
            print(f"{bot_name}: Sorry, I'm just a mental consultant chatbot, this is out of the knowledge range of mine.")
            continue
            
        # Bắt Tình huống Khẩn Cấp
        if tag_name == "crisis_suicide":
            print(f"{bot_name}: Your situation is on THE RED ALERT. Please contact 113 or 115 right now or I'll call them for you.")
            continue
            
        # Tìm kiến thức y khoa
        medical_context = ""
        for intent in intents_data['intents']:
            if tag_name == intent["tag"]: 
                medical_context = random.choice(intent['responses'])
                break
                
        # Gọi Llama-3.2 qua API
        print(f"*(Meiritsuu đang suy nghĩ dựa trên chuẩn đoán: {tag_name}...)*")
        final_answer = ask_llm(sentence, medical_context)
        print(f"{bot_name}: {final_answer}")
        
    else:
        print(f"{bot_name}: Tôi chưa hiểu rõ ý bạn lắm (Mức tự tin chỉ đạt {prob:.2f}). Bạn có thể nói cụ thể hơn được không?")